# U2-L3：IMU 与里程计

**课程目标**

1. 能说明加速度计与陀螺仪各测什么，姿态估计为什么可结合两者的读数。
2. 能说明里程计的位置如何估计、长期运行时为什么可能偏离实际位置。
3. 能用 `get_imu()`、`get_odom()`、`reset_odom()` 读数并按固定间隔连续采样，完成一次直线行走，比较指令距离、里程计读数与卷尺实测三个数。
4. 能算出目标在里程坐标系里的位置对应「在我的前方多远、左侧多远」。


## 课程大纲映射（审核用）

| 轻任务 | 对应位置 | 核心概念 | 轻量概念 |
| --- | --- | --- | --- |
| ③ 姿态实验：读取静止基准，扶稳并缓慢侧倾，比较姿态 | 实验一 4.2、4.3 | 姿态融合 | C09 定时采样 |
| ④ 里程实验：按 0.2 m/s、2 秒行走，比较指令、估计与实测距离 | 实验二 5.1–5.5 | A11 坐标变换：odom → 机器人坐标 | — |

- 姿态角取 `odom.rpy`；加速度和角速度取 `get_imu()`；固定间隔采样用普通循环实现。
- 坐标换算包含真实读数计算与非零 yaw 假设练习，不增加转身运动。
- 接口覆盖：`get_imu()`、`get_odom()`、`reset_odom()`、`set_velocity()`、`set_mode()`。

> 本映射块用于大纲审核，导出学生 PDF 时移除。

## 一、IMU 与姿态角

上一课读过加速度和角速度。它们与姿态有关，但数据含义不同，不能把加速度值直接当作角度。

### 1.1 加速度计与陀螺仪

| 元件 | 读数 | 单位 | 用于估计姿态时的限制 |
| --- | --- | --- | --- |
| 加速度计 | 三个轴的加速度读数；静止且受到支撑时，通常可以看到重力对应的分量 | m/s² | 静止或缓慢运动时可辅助判断倾斜；运动加速度会干扰这种判断 |
| 陀螺仪 | 绕三个轴的角速度，即转动快慢 | rad/s | 角速度随时间累加可估计角度变化，但偏差可能随时间积累 |

手机可利用加速度计判断横屏或竖屏，这是根据重力方向判断倾斜的常见例子；不同设备可能使用不同传感器组合。

### 1.2 姿态融合

姿态估计可结合陀螺仪对转动变化的测量，以及加速度计在合适条件下提供的重力方向信息。利用不同测量的互补关系估计姿态，称为**姿态融合**。重力方向可以约束倾斜，但不能单独确定水平面内的绝对航向，因此 yaw 仍可能有漂移。

本课在 K1 上从 `get_imu()` 读取加速度和角速度，从 `get_odom().rpy` 读取姿态角。IMU 返回结构中虽有 `orientation`、`rpy`，但实际数据可能为空。上述读取方式不表示我们已经掌握 K1 内部融合算法，也不能将所有 IMU 读数一概称为未经处理的原始数据。

### 1.3 RPY 三个角

| 姿态角 | 绕的轴 | 对应的动作 |
| --- | --- | --- |
| roll 横滚 | 前后方向的轴 | 身体向左侧 / 右侧倾斜 |
| pitch 俯仰 | 左右方向的轴 | 上身前倾 / 后仰 |
| yaw 航向 | 上下方向的轴 | 面朝方向改变 |

`odom.rpy` 按 roll、pitch、yaw 排列，单位为弧度。乘 180°/π 可换成度；代码中用 `math.degrees()` 完成换算。

## 二、里程计与漂移

### 2.1 位置估计

里程计给出相对某个参考起点的位置和姿态。关节编码器可以测量转角，结合机器人结构、足部接触等信息，可估计身体相对地面的运动；惯性测量等信息也可参与估计。具体使用哪些数据、怎样融合，取决于机器人实现。

把一段段运动变化累积起来，可得到相对起点的位置。这是理解里程计的基本思路，不能简化成 K1 只按“编码器转角换步长”计算。`position` 的 x、y、z 单位为米。

### 2.2 漂移

传感器偏差、脚底打滑、接触状态判断等因素可能使位置估计逐渐偏离实际位置，这种随时间或运动过程产生的偏离称为**漂移**。误差可能累积，也可能部分抵消；不能说每一步的误差都必然增大。

一次短距离实验可以比较估计与实测的差值。要判断漂移怎样随时间或路程变化，还需要多段、多次测量及一致的测量方法。

### 2.3 把当前位置设为原点

`reset_odom()` 重设里程计参考起点。之后读到的 x、y 表示相对该参考系的位置变化。清零不会提高传感器精度，也不会消除下一段运动产生的误差。

## 三、两个坐标系

机器人在同一时刻可以用两套坐标描述自己的位置，混用就会出错。

| 坐标系 | 原点在哪 | x 轴朝哪 | 用在哪 |
| --- | --- | --- | --- |
| 里程坐标系（odom） | 机器人出发的那一点 | 出发时机器人面朝的方向，之后不再变 | 里程计报出的 x、y |
| 机器人坐标系 | 机器人自己身上，跟着它走 | 机器人**当前**面朝的方向 | 速度指令 `set_velocity(vx, vy, vyaw)` |

出发时可令两套坐标轴对齐；若只平移而没有转向，它们的方向仍可保持一致。机器人转过身之后，`set_velocity(0.2, 0, 0)` 仍然是「朝我脸朝的方向走 0.2 m/s」，它在里程坐标系里的运动方向要结合当前 yaw 判断。

要把目标在里程坐标系里的位置，换成「在我的前方多远、左侧多远」，做一次换算：

```
dx = 目标x − 自己x
dy = 目标y − 自己y
前方 = dx × cos(yaw) + dy × sin(yaw)
左侧 = −dx × sin(yaw) + dy × cos(yaw)
```

`左侧` 算出来是负数，说明目标在右边。前方为负则表示目标在后方。这里讨论水平面内的换算，不考虑高度。

## 四、实验一：姿态观测

**实验名称**：姿态观测

**实验目标**：读出 K1 静止时的加速度与姿态角；按固定间隔连续采样，观察缓慢倾斜机身时 RPY 的变化。

⚠️ **安全**：本课有两次实验。

- **本次姿态观测**只读传感器数据，不发送运动指令，机器人保持阻尼模式（DAMP）下的坐姿、趴姿，或准备模式（PREP）下的站姿，<font color="red">不要进入行走模式（WALK）</font>。
- **下一次里程实验**需要机器人行走。行走的安全要求：机器人必须进入行走模式（WALK）；停止时先切回准备模式（PREP）站稳，**严禁从行走模式（WALK）直接切阻尼模式（DAMP）**（站立状态下会瘫倒）。行走场地需留出 3 米以上直线空间，全程在教师监督下进行。

**前置准备**：K1 已开机并与电脑建立连接（见 U1-L1）；已完成 U1-L2（模式切换与安全）与 U2-L2（传感器速览）。

### 4.1 连接机器人


In [ ]:
from boosteros.robots.booster import BoosterRobot

robot = BoosterRobot()
print("已连接 K1")

### 4.2 静止时的读数

机器人安静地坐着或趴着，先读一组数当基准。


In [ ]:
import math

imu = robot.get_imu()
odom = robot.get_odom()

roll, pitch, yaw = odom.rpy
ax, ay, az = imu.linear_acceleration
wx, wy, wz = imu.angular_velocity

print(f"线加速度 (m/s²)：x={ax:6.2f}  y={ay:6.2f}  z={az:6.2f}")
print(f"角速度   (rad/s)：x={wx:6.3f}  y={wy:6.3f}  z={wz:6.3f}")
print()
print(f"roll  横滚 {math.degrees(roll):7.2f}°")
print(f"pitch 俯仰 {math.degrees(pitch):7.2f}°")
print(f"yaw   航向 {math.degrees(yaw):7.2f}°")

静止时角速度通常接近 0。加速度三个分量的合成大小通常接近 9.8 m/s²，但不一定有某一个轴恰好等于 9.8；机身倾斜时，重力对应的分量会分布到不同轴上。

记录当前姿势下的 RPY 作为基准。坐姿或趴姿的倾角可能较大，不能要求三个角都接近 0，也不把 yaw 当作地理方位角。

### 4.3 按固定间隔连续采样

下面采集 10 轮，每轮读取后等待 1 秒，因此间隔略长于 1 秒。它能帮助理解定时采集的目的，但这里使用的是 Python 循环，并没有创建 ROS2 定时器。

运行这一格的同时，两名同学配合做动作：

1. 教师先确认姿势、接触面和操作空间；保持坐姿或趴姿，一人扶稳躯干两侧，不提离支撑面；
2. 另一人缓慢把机身向左侧倾斜，大约 15 度，停住 1 秒；
3. 再缓慢扶正，最后向右侧倾斜一次，再扶正。

<font color="red">倾斜要慢，按教师示范范围操作，幅度不超过 20 度（这是课堂操作上限，不是摔倒判定阈值），全程扶稳，不要让机器人倾倒或跌落。</font>

In [ ]:
import time

print("开始采样，共 10 轮。请按提示缓慢倾斜机器人。")
print()
print(" 序号    acc_x   acc_y   acc_z     roll    pitch      yaw")
print("-" * 58)

for i in range(10):
    imu = robot.get_imu()
    odom = robot.get_odom()
    r, p, y = odom.rpy
    ax, ay, az = imu.linear_acceleration
    print(f"  {i:>2}   {ax:6.2f}  {ay:6.2f}  {az:6.2f} "
          f"{math.degrees(r):8.2f} {math.degrees(p):8.2f} {math.degrees(y):8.2f}")
    time.sleep(1)

把这 10 行与静止基准比较：

- 左右侧倾时，观察 `roll` 相对基准分别往哪个方向变化；不要求绝对值一定跨过 0。
- 若还发生了前后倾斜，`pitch` 也可能变化。
- 缓慢侧倾会改变重力在各轴上的分量。应比较变化前后的数值，不预设 `acc_z` 一定从 9.8 开始减小。
- `yaw` 变化可能与手动动作中夹带转动或估计误差有关，不能凭一次变化判定原因。

扶正后再与基准比较。若没有回到基准，先核对是否真正回到同一姿势；也可能有传感器噪声或估计误差，不能仅凭这一点认定已经发生漂移。

## 五、实验二：里程实验

**实验名称**：直线行走与距离对照

**实验目标**：把里程计清零，让机器人以 0.2 m/s 直线走 2 秒，比较指令距离、里程计读数与卷尺实测三个数，说明三个数各自的含义与差异。

<font color="red">本实验涉及行走。机器人须进入行走模式（WALK）（准备模式〔PREP〕→ 人工拉起 → 行走模式〔WALK〕）；停止时先切准备模式（PREP）站稳，严禁从行走模式（WALK）直接切阻尼模式（DAMP）。场地留出 3 米以上直线空间，并在地面贴一条起点线，备好卷尺。</font>

**前置准备**：已完成本次实验的姿态观测部分；地面平整、无杂物、无障碍。

### 5.1 里程计清零

先读取并确认模式；按“准备模式（PREP）→ 人工拉起并确认直立 → 行走模式（WALK）”的流程准备，再清零。清零后检查位姿是否接近 0，而非要求浮点值严格等于 0。

<font color="red">只有教师确认处于行走模式（WALK）、场地清空后才进行行走。本课只向前走，不增加转身动作。</font>

In [ ]:
robot.reset_odom()
print("清零后的位姿 [x, y, yaw]：", robot.get_odom().pose_2d.round(3))

### 5.2 目标方位的换算

先给定里程坐标系里的目标点，结合机器人实际位姿计算。运行前根据 yaw 预测：前方、左侧是否会直接等于目标坐标？运行后记录机器人的位姿、目标坐标与换算结果。该格只计算，不向目标运动。

In [ ]:
import math

tx, ty = 2.0, 0.5          # 目标在里程坐标系里的位置，可以改

odom = robot.get_odom()
x, y, yaw = odom.pose_2d

dx, dy = tx - x, ty - y
forward = dx * math.cos(yaw) + dy * math.sin(yaw)
left = -dx * math.sin(yaw) + dy * math.cos(yaw)

print(f"机器人当前位姿：x={x:.2f}  y={y:.2f}  yaw={math.degrees(yaw):.1f}°")
print(f"目标位置      ：x={tx:.2f}  y={ty:.2f}")
print(f"换算到机器人身上：前方 {forward:.2f} m，左侧 {left:.2f} m")

刚清零且没有转向时，yaw 接近 0，换算结果可能与目标 x、y 很接近。为了看出转向的影响，下面使用**假设位姿**，不让机器人实际转身。

设机器人在里程坐标系的 (0, 0)，目标为 (1, 0)：

1. 先在纸上画出里程坐标系 x、y 轴和 yaw=90° 时机器人的朝向。预测目标位于机器人哪一侧。
2. 运行下格核对：前方约 0 m，左侧约 -1 m，即在右侧 1 m。非常小的浮点数可近似看作 0。
3. 将 `demo_yaw` 改为 180°，再次先预测再运行，记录结果并解释“前方为负”的含义。

`yaw` 的正方向按水平面内逆时针计算，公式中的角度必须是弧度。

In [ ]:
# 假设位姿的坐标练习；不读取或控制机器人
import math

demo_x, demo_y = 0.0, 0.0
demo_tx, demo_ty = 1.0, 0.0
demo_yaw = math.radians(90)  # 任务：再改成 180，先预测再运行

demo_dx, demo_dy = demo_tx - demo_x, demo_ty - demo_y
demo_forward = demo_dx * math.cos(demo_yaw) + demo_dy * math.sin(demo_yaw)
demo_left = -demo_dx * math.sin(demo_yaw) + demo_dy * math.cos(demo_yaw)
print(f"前方 {demo_forward:.2f} m，左侧 {demo_left:.2f} m")

### 5.3 直线走 2 秒

本格依次发出前进指令、等待 2 秒、发出零速度指令。不能依赖 Notebook 中断按钮完成停止，因为中断可能发生在零速度指令执行之前。教师应在运行前确认现场停止手段与操作分工；若发生中断，立即按既定停止流程处理，并确认速度归零。

In [ ]:
robot.reset_odom()                   # 再设置一次，防止后面程序执行之前移动过机器人
print("清零后的位姿 [x, y, yaw]：", robot.get_odom().pose_2d.round(3))

In [ ]:
import time

robot.set_velocity(0.2, 0.0, 0.0)    # 以 0.2 m/s 向前
time.sleep(2.0)                      # 持续 2 秒
robot.set_velocity(0.0, 0.0, 0.0)    # 速度归零，停下

print("已停止。指令距离 = 0.2 m/s × 2 s = 0.40 m") 

### 5.4 三个距离的对照

先读里程计，再用卷尺测量水平面内沿起始 x 方向的位移。出发前与停止后均选用同一个参考点，例如躯干中心在地面的投影；不要起点量脚尖、终点改量脚跟。记录方法和读数精度。这里比较的是位移，不是双脚走过的路径总长。

已清零且沿起始方向直行时，可用 odom 的 x 估计前进位移，同时记录 y 和 yaw 检查偏移。

In [ ]:
odom = robot.get_odom()
x, y, yaw = odom.pose_2d

print(f"里程计 x（前进方向）：{x:.3f} m")
print(f"里程计 y（横向偏移）：{y:.3f} m")
print(f"航向 yaw          ：{math.degrees(yaw):.1f}°")
print()
print("把卷尺量出的实距填进记录表，再算里程计与实距的差。")

三个数分别来自不同途径：

- **指令距离 0.40 m**：速度设定值 × 持续时间，是按指令计算的理论值，不代表实测位移。
- **里程计 x**：机器人估计的沿里程坐标系 x 方向的位移。
- **卷尺实测**：按选定参考点测得的位移，也有读数和参考点选择带来的测量误差。

保留一组真机记录供比较，各组应填写自己的实际结果：

| 指令距离 | 里程计读数 | 卷尺实测 |
| --- | --- | --- |
| 0.40 m | 0.776 m | 0.78 m |

这组里程计与卷尺的差约为 -0.004 m。考虑卷尺读数精度，可说两者本次测量较接近；不能据此证明长期没有漂移，也不能把这个差值全归因于里程计。

实际位移与指令理论值差别较大，说明速度设定和实际运动之间存在差异，可能与控制响应、起停过程、地面等因素有关。速度可以通过位移与时间等方式测量，本次记录不足以确定差异的具体原因。

`y` 非零可能反映横向运动、航向偏转或估计误差，需要结合实际轨迹判断，不预先指定唯一原因。

### 5.5 停止收尾

In [ ]:
robot.set_mode("prepare")
print("当前模式：", robot.get_mode())

切回准备模式（PREP）后机器人原地站稳。之后按 U1-L2 学过的收尾流程处理：由人工扶着让机器人坐下，确认坐稳后再进入阻尼模式（DAMP）。


## 六、观察与记录

随各环节填写；坐标练习为假设数据，姿态与行走为现场测量。

| 观察项 | 你的记录 | 说明 |
| --- | --- | --- |
| 静止基准 |  | RPY（度）、三个加速度分量、角速度 |
| 左侧倾与右侧倾 |  | roll 分别相对基准怎样变化 |
| 加速度分量变化 |  | 哪些轴发生变化，是否与倾斜方向有关 |
| 扶正后的读数 |  | 是否回到同一姿势，读数与基准差多少 |
| 实际位姿的目标换算 |  | x、y、yaw、目标 tx/ty、forward/left，并解释正负 |
| 假设 yaw=90° |  | 预测、结果及目标方向 |
| 假设 yaw=180° |  | 预测、结果及目标方向 |
| 指令距离 |  | 0.2 m/s × 2 s = 0.40 m |
| 里程计 x、y、yaw |  | 米、米、度 |
| 卷尺实测 |  | 位移、测量参考点和读数精度 |
| 估计与实测之差 |  | 里程计 x 减卷尺读数；不能单次认定为漂移增长 |

## 七、检查点、思考题与学习评价


### 7.1 分步检查点

- [__] 用 `get_imu()` 读取加速度、角速度，用 `get_odom().rpy` 读取姿态，说明 RPY 各绕哪个轴。
- [__] 连采 10 轮，区分静止基准与侧倾变化，并照实记录扶正后的读数。
- [__] 清零后检查位姿接近 0，能说明清零不会提高估计精度。
- [__] 根据目标的 odom 坐标与机器人位姿算出前方、左侧距离；解释 90°、180° 两个假设例子的正负与方向。
- [__] 按 0.2 m/s、2 秒完成一次直线行走，比较指令、估计、卷尺测量三个数，不将单次差值全部称为漂移。
- [__] 速度先归零，再由行走模式（WALK）切回准备模式（PREP），人工扶坐后进入阻尼模式（DAMP）。

### 7.2 思考题

课堂结合实测讨论第 2 题，其余题课后完成；不要求为答题追加行走。

1. 结合加速度计与陀螺仪估计倾斜，各自提供什么信息？为什么仅凭重力方向还不能确定绝对航向？
2. 本次 y 是多少？它是否一定表示机器人真的横移了这么远？还需要核对什么？
3. 假设同一程序重复运行，哪些因素会使实测结果不同？若要比较漂移随距离的变化，应怎样保持测量方法一致？
4. 光滑地面更容易打滑，但一次估计与实测之差是否一定更大？你还需要哪些证据？

### 7.3 学习评价

1. 一次里程计 x 与卷尺读数之差，可以全部解释为里程计漂移。（判断）
2. 加速度计与陀螺仪分别测量 ______ 与 ______；本课读取姿态角使用 `get_odom()` 返回值的 ______ 属性。
3. 机器人在 odom 的 (0, 0)，yaw=90°，目标在 (1, 0)。目标在机器人的前方约 ______ m、左侧约 ______ m；用自然语言说明目标方向。若 yaw 改为 180°，方向又是什么？

> 参考答案（教师留存，导出学生 PDF 时移除）：
> 1. ×，还包含卷尺测量、参考点等误差，且单次差值不能说明随时间的变化。
> 2. 加速度；角速度；`rpy`。
> 3. 0；-1；右侧 1 m。yaw=180° 时前方 -1 m、左侧约 0，即后方 1 m。

## 八、扩展阅读：定位靠什么兜底

需要长期、稳定定位的室内机器人，常会结合其他信息来校正里程计。展厅、仓库、工厂里常见的做法：

| 做法 | 在场地里加什么 | 机器人怎么用它 |
| --- | --- | --- |
| 地面标记 | 贴二维码、铺磁条、画引导线 | 通过已知位置的标记辅助定位，或沿引导线限定路线；仅有引导线不一定能确定完整位置 |
| 无线基站 | 在墙角装 UWB 基站 | 测出与几个基站的距离，算出自己的位置 |
| 环境地图 | 事先用激光雷达扫一遍场地 | 运行时拿实时扫描和地图比对，反推位置 |

这些方案都要在场地里先做一遍部署和标定，机器人跑起来之后还要反复调。

负责这项工作的岗位通常叫**部署工程师或调试工程师**，日常就是拿卷尺量误差、改参数、挪基站，直到机器人每天跑同一条路线都能停在同一个位置。

本课的距离对照是定位检查的初步练习。现场还需结合任务要求、多次测量与一致的参考点，判断精度是否满足使用需要。